In [4]:
!pip install transformers datasets torch scikit-learn pandas

In [5]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from torch.utils.data import Dataset


In [6]:
texts= [

    # Agriculture
    "crop disease in wheat field",
    "fertilizer needed for crops",
    "pest problem in crops",
    "fasal mein bimari hai",
    "gehu mein disease hai",
    "kheti mein problem ho rahi hai",
    "soil testing required",
    "irrigation issue in field",

    # Government
    "PM Kisan Yojna",
    "government scheme information",
    "kisan registration problem",
    "kisan yojna kya hai",
    "PM Kisan ke baare mein batao",
    "yojna ka registration kaise kare",
    "ration card application",
    "government subsidy details",

    # Electricity
    "electricity problem in village",
"power cut problem",
    "no electricity in house",
    "bijli nahi aa rahi",
    "light chali gayi",
    "bijli problem hai",
    "electricity bill issue",
    "transformer not working",

    # Plumbing
    "water supply issue",
    "pipe leakage",
    "tap not working",
    "paani leak ho raha hai",
    "pipe se paani aa raha hai",
    "paani ki problem hai",
    "water tank problem",
    "drinking water shortage",

    # Healthcare
    "doctor not available",
    "hospital information",
    "medicine required",
    "doctor nahi hai village mein",
    "health center closed",
    "vaccination information",
    "medical emergency",
    "primary health center issue",

    # Roads
    "road damaged",
 "sadak toot gayi hai",
    "road repair needed",
    "village road problem",
    "road construction request",
    "bridge damaged",
    "road blocked",
    "street repair required",

    # Education
    "school teacher absent",
    "school admission information",
    "student scholarship details",
    "school facilities issue",
    "teacher not coming",
    "education scheme information",
    "midday meal issue",
    "school building damaged",

    # Sanitation
    "garbage collection issue",
    "public toilet not working",
    "village cleanliness problem",
    "drainage issue",
    "dirty streets",
    "waste disposal problem",
    "sewer blockage",
    "sanitation complaint"
]
labels = [

    # Agriculture
    "Agriculture","Agriculture","Agriculture","Agriculture",
    "Agriculture","Agriculture","Agriculture","Agriculture",

    # Government
    "Government","Government","Government","Government",
    "Government","Government","Government","Government",

    # Electricity
    "Electricity","Electricity","Electricity","Electricity",
    "Electricity","Electricity","Electricity","Electricity",

    # Plumbing
    "Plumbing","Plumbing","Plumbing","Plumbing",
    "Plumbing","Plumbing","Plumbing","Plumbing",

    # Healthcare
    "Healthcare","Healthcare","Healthcare","Healthcare",
    "Healthcare","Healthcare","Healthcare","Healthcare",

    # Roads
    "Roads","Roads","Roads","Roads",
    "Roads","Roads","Roads","Roads",

   # Education
"Education","Education","Education","Education",
"Education","Education","Education","Education",

# Sanitation
"Sanitation","Sanitation","Sanitation","Sanitation",
"Sanitation","Sanitation","Sanitation","Sanitation"
]

In [7]:
print(len(texts))
print(len(labels))

64
64


In [8]:
label_map = {
    "Agriculture":0,
    "Government":1,
    "Electricity":2,
    "Plumbing":3,
    "Healthcare":4,
    "Roads":5,
    "Education":6,
    "Sanitation":7
}

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df["label"] = df["label"].map(label_map)

df.head()

,text,label
0,crop disease in wheat field,0
1,fertilizer needed for crops,0
2,pest problem in crops,0
3,fasal mein bimari hai,0
4,gehu mein disease hai,0


In [9]:
#splitting the dataset i

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(train_texts))
print("Testing Samples:", len(test_texts))

Training Samples: 51
Testing Samples: 13


In [10]:
#Loading BERT Tokenizer t in2 tokens

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)

print("Tokenizer Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [11]:
#Covting text queries .....tt u by b

train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

print("Tokenization Complete")

Tokenization Complete


In [12]:
class RuralDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [14]:
train_dataset = RuralDataset(
    train_encodings,
    train_labels
)

test_dataset = RuralDataset(
    test_encodings,
    test_labels
)

print("Dataset Ready")

Dataset Ready


In [15]:
#Loads pretrained BERT 4 classi

model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=8
)

print("Model Loaded Successfully")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded Successfully


In [16]:
#Defined how BERT will learn from the rural query dataset


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,
    logging_dir="./logs",
    logging_steps=5
)

print("Training Arguments Ready")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Arguments Ready


In [17]:
# CREATE BERT TRAINER
#Trainer combining all three dsets, p,b

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Trainer Ready")

Trainer Ready


In [18]:
#Training the BERT Model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,2.207054
10,2.066875
15,2.065680
20,1.910831
25,1.887365
30,1.759388
35,1.748959
40,1.697409
45,1.708018
50,1.251307


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=65, training_loss=1.7172624147855318, metrics={'train_runtime': 150.0853, 'train_samples_per_second': 1.699, 'train_steps_per_second': 0.433, 'total_flos': 1310486983200.0, 'train_loss': 1.7172624147855318, 'epoch': 5.0})

In [19]:
!pip install -U accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [20]:
#model eval

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
1.391133,2.121368,65


{'eval_loss': 2.121367931365967}

In [21]:
#Testing Custom Queries

test_query = "bijli nahi aa rahi"

inputs = tokenizer(
    test_query,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

reverse_label_map = {v:k for k,v in label_map.items()}

print("Predicted Category:",
      reverse_label_map[prediction.item()])

Predicted Category: Electricity


In [22]:
recommendations = {

    "Electricity":
    "Contact State Electricity Board or file complaint online.",

    "Agriculture":
    "Check PM-KISAN, crop insurance and agriculture department services.",

    "Healthcare":
    "Visit nearest Primary Health Center or government hospital.",

    "Roads":
    "Contact Municipal Corporation or Public Works Department.",

    "Education":
    "Check scholarship and school services portal.",

    "Government":
    "Visit Common Service Center (CSC) for government schemes.",

    "Plumbing":
    "Contact local water supply department.",

    "Sanitation":
    "Contact municipal sanitation department."
}

In [23]:
predicted_category = reverse_label_map[prediction.item()]

print("Category:", predicted_category)

print("Recommendation:",
      recommendations[predicted_category])

Category: Electricity
Recommendation: Contact State Electricity Board or file complaint online.


In [24]:
service_mapping = {

    "Electricity": {
        "Department":
        "State Electricity Board",

        "Portal":
        "https://pgportal.gov.in",

        "Service":
        "Electricity Complaint Registration"
    },

    "Agriculture": {
        "Department":
        "Agriculture Department",

        "Portal":
        "https://pmkisan.gov.in",

        "Service":
        "PM-KISAN Registration"
    },

    "Education": {
        "Department":
        "Education Department",

        "Portal":
        "https://scholarships.gov.in",

        "Service":
        "Scholarship Information"
    },

    "Healthcare": {
        "Department":
        "Health Department",

        "Portal":
        "https://abdm.gov.in",

        "Service":
        "Ayushman Bharat Services"
    },

    "Roads": {
        "Department":
        "Public Works Department",

        "Portal":
        "https://pgportal.gov.in",

        "Service":
        "Road Damage Complaint"
    },

    "Sanitation": {
        "Department":
        "Municipal Sanitation Department",

        "Portal":
        "https://swachhbharatmission.gov.in",

        "Service":
        "Sanitation Complaint"
    },

    "Government": {
        "Department":
        "Citizen Service Center",

        "Portal":
        "https://www.india.gov.in",

        "Service":
        "Government Schemes"
    },

    "Plumbing": {
        "Department":
        "Water Supply Department",

        "Portal":
        "https://jalshakti-dowr.gov.in",

        "Service":
        "Water Supply Complaint"
    }
}

In [25]:
service = service_mapping[predicted_category]

print("Category:",
      predicted_category)

print("Department:",
      service["Department"])

print("Service:",
      service["Service"])

print("Portal:",
      service["Portal"])

Category: Electricity
Department: State Electricity Board
Service: Electricity Complaint Registration
Portal: https://pgportal.gov.in


In [26]:
def get_help(query):

    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1)

    category = reverse_label_map[prediction.item()]

    service = service_mapping[category]

    print("Query:", query)
    print("Category:", category)
    print("Department:", service["Department"])
    print("Service:", service["Service"])
    print("Portal:", service["Portal"])

In [27]:
get_help("PM Kisan registration problem")

Query: PM Kisan registration problem
Category: Government
Department: Citizen Service Center
Service: Government Schemes
Portal: https://www.india.gov.in


In [30]:
get_help("bijli nahi aa rahi")
get_help("scholarship apply karni hai")
get_help("PM Kisan registration problem")

Query: bijli nahi aa rahi
Category: Electricity
Department: State Electricity Board
Service: Electricity Complaint Registration
Portal: https://pgportal.gov.in
Query: scholarship apply karni hai
Category: Education
Department: Education Department
Service: Scholarship Information
Portal: https://scholarships.gov.in
Query: PM Kisan registration problem
Category: Government
Department: Citizen Service Center
Service: Government Schemes
Portal: https://www.india.gov.in


# Sample Predictions

The following examples demonstrate how the trained multilingual BERT model classifies rural citizen queries into relevant service categories and recommends the appropriate government department, service, and portal.

# Future Improvements

* Increase dataset size for better classification accuracy.
* Support additional regional languages and dialects.
* Integrate real-time government service APIs.
* Deploy as a web or mobile application.
*   Extend the platform with computer vision and intelligent automation features.
* Enhance recommendations using Generative AI assistance.


# Project Conclusion



This project uses multilingual BERT to classify rural citizen queries and recommend relevant government services. It aims to simplify access to public services and can be further enhanced with AI-powered infrastructure inspection and workflow automation.